# Multi-Disease Medical AI — Module 2: Skin Lesion Model Training & Grad-CAM
### ECE Major Project
**Modality:** Dermoscopy (DermaMNIST / HAM10000, $224 \times 224$)
**Objective:** 7-Class Dermoscopic Lesion Classification (Melanoma, Nevus, Basal Cell Carcinoma, etc.) using EfficientNet-B0 + DullRazor DIP + Grad-CAM Explainable AI.

In [ ]:
!pip install -q medmnist torchvision scikit-learn opencv-python-headless matplotlib tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.models as tv_models
from medmnist import DermaMNIST, INFO
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Compute Device:", device)

In [ ]:
# Label Map for 7 Dermoscopic Classes
info = INFO["dermamnist"]
labels_dict = info["label"]
print("Dermatological Classes:")
for k, v in labels_dict.items():
    print(f"  Class {k}: {v}")

# Transforms with Data Augmentation (Rotation, Horizontal/Vertical Flip for dermoscopy)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = DermaMNIST(split="train", download=True, size=224, transform=train_transform)
val_dataset   = DermaMNIST(split="val", download=True, size=224, transform=test_transform)
test_dataset  = DermaMNIST(split="test", download=True, size=224, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"DermaMNIST Splits -> Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

## Build EfficientNet-B0 Backbone with Class Balancing

In [ ]:
# Pretrained EfficientNet-B0 with custom classifier head
model = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.DEFAULT)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, 128),
    nn.SiLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 7)  # 7 classes
)
model = model.to(device)

# Compute class weights to counter dataset imbalance (Nevi dominance)
train_targets = [int(t[0]) for _, t in train_dataset]
class_counts = np.bincount(train_targets, minlength=7)
class_weights = 1.0 / (class_counts + 1e-5)
class_weights = class_weights / np.sum(class_weights) * 7.0
weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=weight_tensor)
optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

In [ ]:
# Training Routine
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, targets in tqdm(loader, desc="Training Skin Model"):
        images = images.to(device)
        targets = targets.squeeze().long().to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)
    return running_loss / total, correct / total

def evaluate(model, loader):
    model.eval()
    all_preds, all_probs, all_targets = [], [], []
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=-1)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_targets.extend(targets.squeeze().numpy())
    acc = np.mean(np.array(all_preds) == np.array(all_targets))
    return acc, np.array(all_preds), np.vstack(all_probs), np.array(all_targets)

## Grad-CAM Saliency Analysis on Skin Lesions
Examines whether the model focuses on asymmetric pigment networks or irrelevant periphery.

In [ ]:
target_layer = model.features[-1]
print("Grad-CAM Target Layer configured:", target_layer)
torch.save(model.state_dict(), "skin_efficientnet_b0.pth")
print("Skin model weights successfully saved.")